# 16: Run a reproducible experiment sweep

**Level:** Advanced  
**Before you start:** Notebooks 03 and 04.  
**Resources:** CPU unless an optional remote step is enabled.

Turn a research question into several independently identifiable jobs, without losing track of results.

Run each cell in order. All core calculations are written in this notebook.

## 1. Define a small sweep

We will compare two rotation angles. Use the same pattern later for learning rates or noise strengths. Each run gets its own script and receipt. The collection is local until submission is enabled.

In [ ]:
from pathlib import Path

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "flagquantum").is_dir()
)
OUTPUTS = ROOT / "workshops/flagos2026/outputs"
OUTPUTS.mkdir(exist_ok=True)


In [ ]:
import json
import runpy

angles = [0.4, 1.2]
runs = []
for index, angle in enumerate(angles):
    script = OUTPUTS / f"sweep_{index}.py"
    source = f"""import flagquantum as fq

def main():
    angle = {angle!r}
    circuit = fq.Circuit(1).ry(0, angle)
    value = fq.run(circuit, outputs=fq.expectation(fq.Z(0))).expectation().item()
    return {{"angle": angle, "z": value}}
"""
    script.write_text(source)
    local = runpy.run_path(str(script))["main"]()
    runs.append(
        {
            "script": str(script),
            "receipt": str(OUTPUTS / f"sweep_{index}.receipt.json"),
            "local": local,
        }
    )
print(json.dumps(runs, indent=2))


## 2. Submit only when the resource budget is agreed

This cell creates at most two CPU jobs, sequentially. A failure leaves earlier receipts intact. Do not rerun the whole submission cell to recover; inspect existing receipts and jobs first.

In [ ]:
from flagquantum.remote.compute.jiuding import JiudingClient

client = JiudingClient()
submit_sweep = False
image = ""  # Instructor-provided compatible image.
if submit_sweep:
    if not image:
        raise ValueError("Provide an image first")
    if any(Path(run["receipt"]).exists() for run in runs):
        raise RuntimeError(
            "Existing receipts found. Reconcile those jobs before submitting."
        )
    for run in runs:
        receipt = client.submit(
            run["script"],
            image=image,
            pythonpath=ROOT,
            receipt=run["receipt"],
            cpus=2,
            memory_gib=2,
        )
        print(receipt)
else:
    print("Sweep built and evaluated locally; no jobs submitted.")


## 3. Collect finished work without resubmitting

Check statuses in a separate cell. Waiting with timeout=0 reads an already completed result or reports that it is not ready.

In [ ]:
collect = False
if collect:
    for run in runs:
        path = Path(run["receipt"])
        if not path.exists():
            print("No receipt:", path.name)
            continue
        receipt = json.loads(path.read_text())
        print(client.status(receipt))
        try:
            print(client.result(receipt, timeout=0))
        except TimeoutError:
            print("Not complete yet; check again later.")


## Make it yours

Use two learning rates instead of two angles. Record source revision, seed, software environment, and stopping condition with each result. Design a table that distinguishes failed, cancelled, running, and completed experiments. Cancel unwanted jobs explicitly using their receipts.